In [1]:
import random
import copy
import torch
import pickle
import os
import matplotlib.pyplot as plt
import numpy as np

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import *
from causal_rl.algo.imitation.gail.causal_gail import *
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "figure.dpi": 150,
})

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
hidden/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000

In [4]:
# for eval: corrupted W, O shown
eval_env = AntMazePCH(env_id='antmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True)

In [5]:
# load model
MODEL_PATH = 'hidden/antmaze_large_expert_finetuned.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

action_bounds = (ckpt['action_bounds_low'], ckpt['action_bounds_high'])

expert_model = ContinuousPolicyNN(
    input_dim=ckpt['input_dim'],
    action_dim=ckpt['num_actions'],
    hidden_dim=ckpt['hidden_dim'],
    num_blocks=ckpt['num_blocks'],
    dropout=ckpt['dropout'],
    layernorm=ckpt['layernorm'],
    final_tanh=ckpt['final_tanh'],
    action_bounds=action_bounds,
).to(device)

expert_model.load_state_dict(ckpt['state_dict'])
expert_model.eval()

slots = ckpt['slots']
Z_trim = ckpt['Z_trim']
dims = ckpt['dims']
lookback = ckpt['lookback']

expert_policy = shared_policy_fn_long_horizon(expert_model, slots, Z_trim, continuous=True, device=device)
expert_policies = make_shared_policy_dict(expert_policy)

In [6]:
num_eval_eps = 500

expert_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=expert_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    show_progress=True
)

len(expert_returns)

Starting episode 1/500...


  Episode 1 ended at step 1000 (terminated: False, truncated: True).
Starting episode 2/500...


  Episode 2 ended at step 722 (terminated: True, truncated: False).
Starting episode 3/500...


  Episode 3 ended at step 570 (terminated: True, truncated: False).
Starting episode 4/500...


  Episode 4 ended at step 1000 (terminated: False, truncated: True).
Starting episode 5/500...


...

  Episode 497 ended at step 556 (terminated: True, truncated: False).
Starting episode 498/500...


  Episode 498 ended at step 597 (terminated: True, truncated: False).
Starting episode 499/500...


  Episode 499 ended at step 776 (terminated: True, truncated: False).
Starting episode 500/500...


  Episode 500 ended at step 483 (terminated: True, truncated: False).
Finished collecting imitator trajectories.


401449

In [7]:
with open('hiddenexpert_traj_antlarge.pkl', 'wb') as f:
    pickle.dump(expert_returns, f)

print(f'saved {len(expert_returns)} trajectories')

saved 401449 trajectories
